# Bronze -- `bronze_adventure_works_sales`

Landing only. No deduplication, no filtering, no business logic.

**Source:** `adventure_works.sales`  
**File:** `Sales.csv`  
**Watermark:** `None`  
**Load pattern:** `full`

> Every upstream defect survives this layer intact. If bronze silently fixed anything, a silver bug would be indistinguishable from an upstream change and replay would not reproduce the original state.

> GENERATED FILE -- DO NOT EDIT.
Produced by framework/generators/generate_notebooks.py from the project spec set. Edit the spec and regenerate; hand edits are overwritten and will fail the notebook-lint gate.


In [ ]:
# Parameters -- overridden per environment by the deployment pipeline.
# See 05-deployment.yaml `parameterisation`.
target_item = "lh_bronze"
source_item = "lh_bronze"
environment = "dev"
dq_failure_action = "warn"

import sys
import re
from datetime import datetime

from pyspark.sql import functions as F

from ttfabric.cleansing import RuleContext, get_rule
from ttfabric.quality import DQRunLog

load_id = f"load_{datetime.utcnow():%Y%m%d_%H%M%S}"

def resolve_table(name: str):
    """Resolve a spec table reference to a DataFrame.

    Deliberately UNQUALIFIED, so the read lands in the default lakehouse.

    Rules reference tables in their OWN layer -- enforce_referential_integrity
    against dim_products, recompute_total_from_lines against fct_order_items --
    and those peers live in the item this notebook writes to, not the one it
    reads its source from. Qualifying with source_item sent them to
    lh_bronze.dim_products, which does not and should not exist.

    The single cross-item read, this table's own bronze source, is qualified
    explicitly at the call site instead.
    """
    bare = name.split(".")[-1]
    return spark.read.table(bare)

ctx = RuleContext(
    spark=spark,
    load_id=load_id,
    environment=environment,
    table="bronze_adventure_works_sales",
    resolve_table=resolve_table,
    apply_masking=(environment in ("uat", "prod")),
)

dq = DQRunLog(spark, load_id=load_id, layer="bronze", table_name="bronze_adventure_works_sales")
print(f"load_id={load_id}  environment={environment}  table=bronze_adventure_works_sales")

In [ ]:
# ---- Declared schema ---------------------------------------------
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, LongType, BooleanType, DateType)

schema = StructType([
    StructField("sales_order_number", StringType(), True),
    StructField("order_date", DateType(), True),
    StructField("product_key", IntegerType(), True),
    StructField("reseller_key", IntegerType(), True),
    StructField("employee_key", IntegerType(), True),
    StructField("sales_territory_key", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", StringType(), True),
    StructField("sales", StringType(), True),
    StructField("cost", StringType(), True),
])


In [ ]:
# ---- Read the landed file ----------------------------------------
landing_path = "Files/bronze/adventure_works/sales"

def normalize_header(name: str) -> str:
    name = name.replace(" ", "_").replace("-", "_")
    s1 = re.sub("(.)([A-Z][a-z]+)", r"\1_\2", name)
    result = re.sub("([a-z0-9])([A-Z])", r"\1_\2", s1).lower()
    return re.sub(r"_+", "_", result)

# Read with tab delimiter (handles current format)
df_raw = (spark.read
    .option("header", "true")
    .option("delimiter", "\t")
    .option("encoding", "utf-8")
    .option("quote", '"')
    .csv(landing_path))

# Rename columns to snake_case
column_mapping = {old: normalize_header(old) for old in df_raw.columns}
df = df_raw
for old_name, new_name in column_mapping.items():
    if old_name != new_name:
        df = df.withColumnRenamed(old_name, new_name)

# Cast to proper types
df = df.select(
    F.col("sales_order_number").cast(StringType()),
    F.col("order_date").cast(DateType()),
    F.col("product_key").cast(IntegerType()),
    F.col("reseller_key").cast(IntegerType()),
    F.col("employee_key").cast(IntegerType()),
    F.col("sales_territory_key").cast(IntegerType()),
    F.col("quantity").cast(IntegerType()),
    F.col("unit_price").cast(StringType()),
    F.col("sales").cast(StringType()),
    F.col("cost").cast(StringType()),
)

rows_in = df.count()
dq.record_input(rows_in)
print(f"read {rows_in:,} rows from {landing_path}")

In [ ]:
# ---- Landing checks ----------------------------------------------
# These test whether the file ARRIVED correctly -- not whether its
# contents are any good, which is silver's job.
#
# The header is re-read WITHOUT the declared schema. Reading it from
# `df.columns` would return the schema's own names, so the check would
# compare the schema to itself and could never fail -- while the
# schema, applied positionally, silently loaded values into the wrong
# columns. That failure is invisible whenever the mismatched columns
# share a type.
expected = [f.name for f in schema.fields]
actual = (spark.read
    .option("header", "true")
    .csv(landing_path)
    .columns)

if actual != expected:
    raise AssertionError(
        f"header_matches_registry failed.\n"
        f"  registry: {expected}\n"
        f"  file:     {actual}\n"
        f"Order matters -- the schema is applied positionally."
    )
if rows_in == 0:
    raise AssertionError("row_count_not_zero failed: a zero-row file "
                         "almost always means a broken export")
print("landing checks passed")


In [ ]:
# ---- Audit columns and write -------------------------------------
# Injected from 00-platform.yaml `audit_columns.bronze`, so the
# provenance contract is identical across every entity.
out = (df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source", F.lit("adventure_works"))
    .withColumn("_load_id", F.lit(load_id))
    .withColumn("_source_file", F.input_file_name())
)

# `ingest_date` is the partition a full snapshot owns. Stamped
# here rather than read from the landing path, so a re-run on the
# same day targets the same partition.
out = out.withColumn("ingest_date", F.current_date())

# IDEMPOTENT per ingest_date. Not plain append: a pipeline retry
# would otherwise land a second full snapshot and report success.
# No `overwriteSchema` here. Delta rejects it outright in dynamic
# partition overwrite mode -- DELTA_OVERWRITE_SCHEMA_WITH_DYNAMIC_
# PARTITION_OVERWRITE -- and it would be wrong regardless: F3 sets
# on_schema_drift: fail, so a changed source schema must STOP the
# load rather than quietly rewrite the table around it.
(out.write.mode("overwrite")
    .option("partitionOverwriteMode", "dynamic")
    .partitionBy("ingest_date")
    .format("delta").saveAsTable("bronze_adventure_works_sales"))

dq.record_output(rows_in)
print(f"landed {rows_in:,} rows into bronze_adventure_works_sales")
dq.flush()
